In [10]:
from google.colab import userdata
key = userdata.get("GEMINI_API_KEY")
print("key loaded:", bool(key))

key loaded: True


In [11]:
# Cell 1 lets you ask Gemini a question from the notebook, and it saves every answer in a file.

!pip install -q google-genai

import os, json, time, hashlib
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-3.5-flash-lite"
CACHE_PATH = "gemini_cache.json"
cache = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}
last_call = [0.0]

def call_model(prompt):
    key = hashlib.sha256((MODEL + "||" + prompt).encode()).hexdigest()
    if key in cache:
        return cache[key]
    wait = 13 - (time.time() - last_call[0])
    if wait > 0:
        time.sleep(wait)
    resp = client.models.generate_content(model=MODEL, contents=prompt)
    last_call[0] = time.time()
    cache[key] = resp.text
    json.dump(cache, open(CACHE_PATH, "w"))
    return resp.text

print(call_model("Say hello in one short sentence."))

# OUTPUT
# print(call_model(...))  -> function: call_model(prompt)
#                            output: a short greeting written by Gemini
#                            first run: slow (real request), second run: instant (from the file)

#  prompt: "Say hello in one short sentence."
#        │
#        ▼
#  ┌────────────────────────────────────────────┐
#  │ call_model: same prompt asked before?      │
#  └───────────────┬─────────────────┬──────────┘
#                  │ yes             │ no
#                  ▼                 ▼
#     return the saved answer   ┌──────────────────────────┐
#     (instant, free)           │ wait a few seconds       │
#                               │ ask Gemini               │
#                               │ save the answer to file  │
#                               └────────────┬─────────────┘
#                                            ▼
#                                  return the answer

Hello there, how can I help you today?


In [ ]:
# Cell 2 compares the skills in an offer with the skills in a CV, and says if the student qualifies, is almost there, is not suited, or if the offer is too vague.

!pip install -q sentence-transformers

import pandas as pd
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2") # no LLm = SentenceTransformer
TAU = 0.50 # threshold Score >= 50 Skill matches.

def match(requirements, cv_skills): # match every requirement with every skill and give a score, -1 <= score <= 1, using sims
    R = model.encode(requirements, normalize_embeddings=True)
    S = model.encode(cv_skills, normalize_embeddings=True)
    sims = R @ S.T
    out = []
    for i, req in enumerate(requirements):
        j = sims[i].argmax()
        out.append((req, cv_skills[j], float(sims[i][j])))
    return out

def decide(results, tau=TAU):
    if not results:
        return "refuse", []  # No requirements to evaluate -> refuse
    missing = [req for req, _, score in results if score < tau]
    verdict = {0: "qualifies", 1: "almost"}.get(len(missing), "not suited")
    return verdict, missing

def label(score): # text next to each score: high >= 0.75, mid >= TAU, low < TAU (low = missing)
    if score >= 0.75:
        return "high"
    if score >= TAU:
        return "mid"
    return "low"

requirements = ["Python", "Requêtes SQL", "Tableaux de bord Power BI", "Anglais professionnel"]
cv_skills = ["Programmation Python", "SQL / PostgreSQL", "Anglais courant (B2)", "Git / GitHub"]

res = match(requirements, cv_skills)
for req, skill, score in res:
    print(f"{req:30} -> {skill:25} {score:.2f}  {label(score)}")
print(decide(res))

# sims: a score for every pair (rows = requirements, columns = CV skills)
R = model.encode(requirements, normalize_embeddings=True)
S = model.encode(cv_skills, normalize_embeddings=True)
sims = R @ S.T
print("\nsims = R @ S.T")
print(pd.DataFrame(sims, index=requirements, columns=cv_skills).round(2))

# OFFER requirements                      CV skills
# ┌───────────────────────────┐           ┌───────────────────────────┐
# │ Python                    │           │ Programmation Python      │
# │ Requêtes SQL              │           │ SQL / PostgreSQL          │
# │ Tableaux de bord Power BI │           │ Anglais courant (B2)      │
# │ Anglais professionnel     │           │ Git / GitHub              │
# └─────────────┬─────────────┘           └─────────────┬─────────────┘
#               │                                       │
#               ▼                                       ▼
#        ┌─────────────────────────────────────────────────────┐
#        │ model.encode: turns every text into numbers         │
#        └──────────────────────────┬──────────────────────────┘
#                                   ▼
#        ┌─────────────────────────────────────────────────────┐
#        │ sims = R @ S.T: a score for every pair              │
#        └──────────────────────────┬──────────────────────────┘
#                                   ▼
#        ┌─────────────────────────────────────────────────────┐
#        │ match(): for each requirement, keep the closest     │
#        │ skill and its score                                 │
#        │                                                     │
#        │ Python         -> Programmation Python    high      │
#        │ Requêtes SQL   -> SQL / PostgreSQL        mid       │
#        │ Power BI       -> (nothing close)         low       │
#        │ Anglais pro    -> Anglais courant (B2)    mid       │
#        └──────────────────────────┬──────────────────────────┘
#                                   ▼
#        ┌─────────────────────────────────────────────────────┐
#        │ decide(): score below TAU (0.50) = missing          │
#        └──────────────────────────┬──────────────────────────┘
#                                   ▼
#        ┌──────────────┬───────────┴──────────┬───────────────┐
#        ▼              ▼                      ▼               ▼
#    0 missing      1 missing             2+ missing     no requirements
#    qualifies        almost              not suited         refuse
#                       ▲
#                       └── this test: only Power BI is missing

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Python                         -> Programmation Python      0.92  high
Requêtes SQL                   -> SQL / PostgreSQL          0.67  mid
Tableaux de bord Power BI      -> Anglais courant (B2)      0.19  low
Anglais professionnel          -> Anglais courant (B2)      0.59  mid
('almost', ['Tableaux de bord Power BI'])

sims = R @ S.T
                           Programmation Python  SQL / PostgreSQL  \
Python                                     0.92              0.00   
Requêtes SQL                               0.21              0.67   
Tableaux de bord Power BI                 -0.02              0.08   
Anglais professionnel                      0.25             -0.08   

                           Anglais courant (B2)  Git / GitHub  
Python                                     0.30          0.11  
Requêtes SQL                               0.04          0.00  
Tableaux de bord Power BI                  0.19          0.13  
Anglais professionnel                      0.59          0.

In [ ]:
# Cell 3 runs the Match step on Habib's CVs and offers, and shows where it disagrees with the exact comparison of skill names.

import json, os
import pandas as pd

REPO_URL = "https://github.com/Jaafar-Trabelsi/internship-match"  # TODO: put the real repo URL (private repo: use a token from Colab Secrets)
if not os.path.exists("internship-match"):
    assert "<user>" not in REPO_URL, "Fill REPO_URL first"
    !git clone -q {REPO_URL}
DATA = "internship-match/data/"

cvs = json.load(open(DATA + "cvs.json", encoding="utf-8"))
offers = json.load(open(DATA + "offers.json", encoding="utf-8"))

def words(tags):
    return [t.replace("_", " ") for t in tags]

rows = []
for c in cvs:
    for o in offers:
        if not o["requirements"]:
            continue
        res = match(words(o["requirements"]), words(c["skills"]))
        verdict, missing = decide(res)
        exact_missing = [r for r in o["requirements"] if r not in c["skills"]]
        exact = {0: "qualifies", 1: "almost"}.get(len(exact_missing), "not suited")
        rows.append((c["id"], o["id"], verdict, exact, "" if verdict == exact else "DIFFERENT"))

df = pd.DataFrame(rows, columns=["cv", "offer", "embeddings", "exact names", "check"])
print(df.to_string(index=False))
print(df["check"].eq("DIFFERENT").sum(), "pairs differ out of", len(df))

# INPUT (uploaded)                       FOR EVERY CV x EVERY OFFER (vague offers skipped)
# ┌──────────────────────┐               ┌──────────────────────────────────────┐
# │ cvs.json   (skills)  │─────────────▶│ words(): machine_learning            │
# │ offers.json (reqs)   │               │          -> machine learning         │
# └──────────────────────┘               └──────────────────┬───────────────────┘
#                                                           │
#                                       ┌───────────────────┴───────────────────┐
#                                       ▼                                       ▼
#                        ┌──────────────────────────────┐        ┌──────────────────────────────┐
#                        │ EMBEDDINGS                   │        │ EXACT NAMES (no AI)          │
#                        │ match() -> score             │        │ is the tag in the CV list?   │
#                        │ decide(): score < 0.50       │        │ count the missing tags       │
#                        │ = missing -> verdict         │        │ -> verdict                   │
#                        └──────────────┬───────────────┘        └──────────────┬───────────────┘
#                                       └───────────────────┬───────────────────┘
#                                                           ▼
#                                       ┌───────────────────────────────────────┐
#                                       │ same verdict? blank                   │
#                                       │ different?    DIFFERENT               │
#                                       └───────────────────┬───────────────────┘
#                                                           ▼
#                                       table of all pairs + "N pairs differ out of 60"
#                                       DIFFERENT = a false match (excel -> sql)
#                                                   or a good one (pytorch -> deep learning)

 cv offer embeddings exact names     check
C01   O01  qualifies   qualifies          
C01   O02 not suited  not suited          
C01   O03 not suited  not suited          
C01   O04 not suited  not suited          
C01   O05  qualifies      almost DIFFERENT
C01   O06 not suited  not suited          
C02   O01     almost  not suited DIFFERENT
C02   O02 not suited  not suited          
C02   O03 not suited  not suited          
C02   O04 not suited  not suited          
C02   O05     almost  not suited DIFFERENT
C02   O06 not suited  not suited          
C03   O01 not suited  not suited          
C03   O02 not suited  not suited          
C03   O03  qualifies      almost DIFFERENT
C03   O04 not suited  not suited          
C03   O05 not suited  not suited          
C03   O06 not suited  not suited          
C04   O01 not suited  not suited          
C04   O02 not suited  not suited          
C04   O03 not suited  not suited          
C04   O04     almost      almost          
C04   O05 n

In [ ]:
# Cell 4 prints, for the 9 pairs marked DIFFERENT, the closest CV skill and score of each requirement that is not written in the CV.

pairs = [("C01","O05"), ("C02","O01"), ("C02","O05"), ("C03","O03"), ("C07","O02"),
         ("C08","O01"), ("C08","O05"), ("C10","O01"), ("C10","O05")]

def show(cv_id, offer_id):
    c = next(x for x in cvs if x["id"] == cv_id)
    o = next(x for x in offers if x["id"] == offer_id)
    print(cv_id, "vs", offer_id)
    res = match(words(o["requirements"]), words(c["skills"]))
    for tag, (req, skill, score) in zip(o["requirements"], res):
        if tag not in c["skills"]:
            print(f"  {req:20} -> {skill:18} {score:.2f}  {label(score)}")

for cv_id, offer_id in pairs:
    show(cv_id, offer_id)

C01 vs O05
  databases            -> sql                0.76  high
C02 vs O01
  powerbi              -> nodejs             0.35  low
  excel                -> sql                0.53  mid
C02 vs O05
  powerbi              -> nodejs             0.35  low
  databases            -> sql                0.76  high
C03 vs O03
  deep learning        -> machine learning   0.68  mid
C07 vs O02
  nextjs               -> css                0.67  mid
C08 vs O01
  powerbi              -> machine learning   0.20  low
  excel                -> sql                0.53  mid
C08 vs O05
  powerbi              -> machine learning   0.20  low
  databases            -> sql                0.76  high
C10 vs O01
  excel                -> sql                0.53  mid
C10 vs O05
  databases            -> sql                0.76  high


In [ ]:
# Cell 5 is the baseline: Gemini alone reads the raw CV and the raw offer and gives a verdict, with no extraction, no embeddings and no rule.

import re

eval_set = json.load(open(DATA + "eval_set.json", encoding="utf-8"))
cv_by_id = {c["id"]: c for c in cvs}
offer_by_id = {o["id"]: o for o in offers}

def baseline(cv_text, offer_text):
    prompt = f"""You compare a student CV with an internship offer.
Answer with one verdict:
- qualifies: the CV covers all requirements of the offer
- almost: exactly one requirement is missing
- not suited: two or more requirements are missing
- refuse: the offer gives too little information to judge

CV:
{cv_text}

OFFER:
{offer_text}

Reply with JSON only, like {{"verdict": "almost", "missing": ["powerbi"]}}"""
    raw = call_model(prompt)
    m = re.search(r"\{.*\}", raw, re.S)
    try:
        return json.loads(m.group(0))
    except Exception:
        return {"verdict": "ERROR", "missing": []}

rows = []
for case in eval_set:
    out = baseline(cv_by_id[case["cv"]]["text"], offer_by_id[case["offer"]]["text"])
    ok = out["verdict"] == case["gold_verdict"]
    rows.append((case["case"], case["cv"], case["offer"], case["gold_verdict"], out["verdict"], "" if ok else "WRONG"))

bdf = pd.DataFrame(rows, columns=["case", "cv", "offer", "gold", "gemini", "check"])
print(bdf.to_string(index=False))
n_ok = (bdf["check"] == "").sum()
ref = bdf[bdf["gold"] == "refuse"]
print(f"\nCorrect verdicts: {n_ok} / {len(bdf)}")
print(f"Correct refusals: {(ref['gemini'] == 'refuse').sum()} / {len(ref)}")

# INPUT (uploaded)                  FOR EACH OF THE 24 CASES
# ┌───────────────────────┐         ┌─────────────────────────────────────────┐
# │ eval_set.json         │───────▶│ take the raw CV text + raw offer text   │
# │ (case, cv, offer,     │         │ (NOT the skills / requirements tags)    │
# │  gold verdict)        │         └────────────────────┬────────────────────┘
# └───────────────────────┘                              ▼
#                                     ┌─────────────────────────────────────────┐
#                                     │ baseline(): prompt = rules of the 4     │
#                                     │ verdicts + CV text + offer text         │
#                                     └────────────────────┬────────────────────┘
#                                                          ▼
#                                     ┌─────────────────────────────────────────┐
#                                     │ call_model(): Gemini answers            │
#                                     │ {"verdict": "almost", "missing": [...]} │
#                                     │ (saved in the cache file)               │
#                                     └────────────────────┬────────────────────┘
#                                                          ▼
#                                     ┌─────────────────────────────────────────┐
#                                     │ compare with the gold verdict           │
#                                     │ same -> blank      different -> WRONG   │
#                                     └────────────────────┬────────────────────┘
#                                                          ▼
#                                     table of 24 cases
#                                     Correct verdicts: X / 24
#                                     Correct refusals: X / 4
#
# NO extraction, NO embeddings, NO rule: Gemini decides everything alone.
# This is the number our full system has to beat.

 case  cv offer       gold     gemini check
    1 C01   O01  qualifies     almost WRONG
    2 C01   O05  qualifies     almost WRONG
    3 C02   O06  qualifies  qualifies      
    4 C06   O02  qualifies  qualifies      
    5 C08   O03  qualifies     almost WRONG
    6 C09   O04  qualifies     almost WRONG
    7 C02   O05     almost not suited WRONG
    8 C03   O03     almost     almost      
    9 C04   O04     almost     almost      
   10 C07   O02     almost     almost      
   11 C08   O05     almost     almost      
   12 C10   O01     almost     almost      
   13 C02   O01 not suited not suited      
   14 C05   O06 not suited not suited      
   15 C01   O02 not suited not suited      
   16 C03   O01 not suited     almost WRONG
   17 C04   O03 not suited not suited      
   18 C09   O03 not suited not suited      
   19 C06   O04 not suited not suited      
   20 C05   O02 not suited not suited      
   21 C01   O07     refuse     refuse      
   22 C06   O08     refuse     r

In [ ]:
from google.colab import files
files.download("gemini_cache.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 6 prints each offer's requirements list next to its full text, to check they say the same thing.

import textwrap

for oid in ["O01", "O05", "O06"]:
    o = offer_by_id[oid]
    print("=" * 70)
    print(f"{oid}  |  {o['title']}")
    print("=" * 70)
    print("REQUIREMENTS (our list):")
    for i, tag in enumerate(o["requirements"], 1):
        print(f"  {i}. {tag}")
    print("\nOFFER TEXT (what Gemini reads):")
    print(textwrap.fill(o["text"], width=70, initial_indent="  ", subsequent_indent="  "))
    print()

O01  |  Marketing Data Analyst Intern - PFE (Bosch, Casablanca)
REQUIREMENTS (our list):
  1. powerbi
  2. sql
  3. excel

OFFER TEXT (what Gemini reads):
  Marketing Data Analyst internship (PFE) at Bosch in Casablanca. The
  intern designs, develops and maintains Power BI dashboards to track
  marketing KPIs and support reporting and decision-making, working
  closely with the Data Manager. Candidates are final-year students in
  engineering, computer science, data or information systems, with
  solid foundations in data analysis, knowledge of Power BI, and
  familiarity with SQL and advanced Excel. Python and basic AI/ML
  knowledge are mentioned as a plus or as motivation to learn.

O05  |  Data & Reporting Intern (Python + Power BI)
REQUIREMENTS (our list):
  1. powerbi
  2. databases

OFFER TEXT (what Gemini reads):
  Data and reporting internship. The intern collects, cleans and
  structures source data, writes Python scripts to automate data
  processing, and designs and update

In [ ]:
bdf.to_csv("baseline_results.csv", index=False)
files.download("baseline_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 7 is the Extract step: Gemini reads the raw text of each offer and each CV and returns a short list of skills.

import re

def extract_skills(text, kind):
    if kind == "offer":
        task = ("List the technical skills and tools that candidates need for this internship offer. "
                "Do not include things described as a plus or nice to have. "
                "If the offer names no concrete requirements, return an empty list.")
    else:
        task = "List the technical skills and tools this student has used or knows."
    prompt = f"""{task}
Use short lowercase English names, for example "power bi", "sql", "python", "react".
Reply with JSON only, like {{"skills": ["python", "sql"]}}

TEXT:
{text}"""
    raw = call_model(prompt)
    m = re.search(r"\{.*\}", raw, re.S)
    try:
        return [s.strip().lower() for s in json.loads(m.group(0))["skills"]]
    except Exception:
        return None

offer_skills = {o["id"]: extract_skills(o["text"], "offer") for o in offers}
cv_skills = {c["id"]: extract_skills(c["text"], "cv") for c in cvs}

print("OFFERS: Gemini extracted  |  our list")
for o in offers:
    print(o["id"], offer_skills[o["id"]], " | ", o["requirements"])
print("\nCVS: Gemini extracted  |  our list")
for c in cvs:
    print(c["id"], cv_skills[c["id"]], " | ", c["skills"])

# OFFER or CV text                 Gemini                         result
# ┌───────────────────────┐        ┌──────────────────────┐       ┌────────────────────────┐
# │ raw text (paragraph)  │───────▶│ extract_skills():    │──────▶│ ["power bi", "sql",    │
# │ from offers / cvs     │        │ "list the skills"    │       │  "excel"]              │
# └───────────────────────┘        └──────────────────────┘       └────────────────────────┘
# vague offer -> Gemini returns []   (this is how the system will know to refuse)

OFFERS: Gemini extracted  |  our list
O01 ['power bi', 'sql', 'excel']  |  ['powerbi', 'sql', 'excel']
O02 ['react', 'typescript', 'next.js', 'python', 'fastapi', 'rest apis']  |  ['react', 'typescript', 'nextjs', 'python', 'fastapi', 'rest_apis']
O03 ['python', 'machine learning', 'deep learning']  |  ['python', 'machine_learning', 'deep_learning']
O04 ['devops', 'infrastructure as code', 'automation', 'networks', 'gnu/linux', 'linux']  |  ['devops', 'networking', 'linux']
O05 ['power bi', 'python', 'sql']  |  ['powerbi', 'databases']
O06 ['html', 'css', 'javascript', 'react', 'angular', 'vue.js', 'python', 'django', 'node.js']  |  ['html', 'css', 'javascript', 'frontend_framework', 'backend']
O07 []  |  []
O08 []  |  []
O09 []  |  []
O10 []  |  []

CVS: Gemini extracted  |  our list
C01 ['power bi', 'python', 'pandas', 'sql', 'postgresql', 'excel']  |  ['python', 'sql', 'powerbi', 'excel', 'pandas', 'statistics']
C02 ['react', 'node.js', 'git', 'github', 'html', 'css', 'javascript', 

In [ ]:
# Cell 8 chains Extract -> Match -> Decide on the 24 evaluation cases, with no Gemini call in Match or Decide.

rows = []
for case in eval_set:
    reqs = offer_skills[case["offer"]] or []
    skills = cv_skills[case["cv"]] or []
    if reqs and skills:
        res = match(reqs, skills)
    else:
        res = []
    verdict, missing = decide(res)
    ok = verdict == case["gold_verdict"]
    rows.append((case["case"], case["cv"], case["offer"], case["gold_verdict"], verdict, missing, "" if ok else "WRONG"))

fdf = pd.DataFrame(rows, columns=["case", "cv", "offer", "gold", "system", "missing", "check"])
print(fdf.to_string(index=False))
n_ok = (fdf["check"] == "").sum()
ref = fdf[fdf["gold"] == "refuse"]
print(f"\nCorrect verdicts: {n_ok} / {len(fdf)}")
print(f"Correct refusals: {(ref['system'] == 'refuse').sum()} / {len(ref)}")
fdf.to_csv("full_system_results.csv", index=False)

# raw text ──▶ EXTRACT (Gemini, done in Cell 7) ──▶ requirements + CV skills
#          ──▶ MATCH (embeddings) ──▶ scores ──▶ DECIDE (rule) ──▶ verdict + missing
#          ──▶ compare with gold ──▶ X / 24

 case  cv offer       gold     system                                                          missing check
    1 C01   O01  qualifies  qualifies                                                               []      
    2 C01   O05  qualifies  qualifies                                                               []      
    3 C02   O06  qualifies not suited                                        [angular, python, django] WRONG
    4 C06   O02  qualifies     almost                                                      [rest apis] WRONG
    5 C08   O03  qualifies  qualifies                                                               []      
    6 C09   O04  qualifies not suited                             [infrastructure as code, automation] WRONG
    7 C02   O05     almost not suited                                               [power bi, python] WRONG
    8 C03   O03     almost  qualifies                                                               [] WRONG
    9 C04   O04    

## v2 and ablation
Extract v2 groups "or" alternatives and ignores skills the student lacks. The v2 prompts were written after looking at the v1 failures on these same 24 cases, so the v2 number is optimistic. The prompt examples are generic, not copied from the evaluation data.

In [9]:
import re, json
import pandas as pd

In [12]:
# Cell 9 is Extract v2: Gemini returns requirement GROUPS (alternatives = one group) and skips skills the student lacks.

def _json(raw):
    m = re.search(r"\{.*\}", raw, re.S)
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def extract_offer_v2(text):
    prompt = f"""List the technical skills and tools that candidates need for this internship offer.
Rules:
- Include every concrete skill or tool the offer names as required or expected.
- If the offer gives alternatives for one need (for example "MySQL or PostgreSQL", "Java, Kotlin or Scala"), put them together in ONE group: the candidate needs only one of them.
- Do not include things described as a plus or nice to have.
- If the offer names no concrete requirements, return an empty list.
Use short lowercase English names, for example "power bi", "sql", "react".
Reply with JSON only, like {{"requirements": [["python"], ["mysql", "postgresql"], ["git"]]}}

TEXT:
{text}"""
    data = _json(call_model(prompt))
    try:
        return [[s.strip().lower() for s in ([g] if isinstance(g, str) else g)] for g in data["requirements"]]
    except Exception:
        return None

def extract_cv_v2(text):
    prompt = f"""List the technical skills and tools this student has used or knows.
Rules:
- Include every concrete skill, tool or concept the text names, including short ones such as "REST APIs".
- Do NOT include anything the text says the student has not used, does not know or lacks (for example "has not used Kubernetes", "no experience with Docker").
Use short lowercase English names, for example "power bi", "sql", "react".
Reply with JSON only, like {{"skills": ["python", "sql"]}}

TEXT:
{text}"""
    data = _json(call_model(prompt))
    try:
        return [s.strip().lower() for s in data["skills"]]
    except Exception:
        return None

offer_groups = {o["id"]: extract_offer_v2(o["text"]) for o in offers}
cv_skills_v2 = {c["id"]: extract_cv_v2(c["text"]) for c in cvs}

print("OFFERS (v2 groups)")
for o in offers:
    print(o["id"], offer_groups[o["id"]])
print("\nCVS (v2 skills)")
for c in cvs:
    print(c["id"], cv_skills_v2[c["id"]])

OFFERS (v2 groups)
O01 [['power bi'], ['sql'], ['excel']]
O02 [['react'], ['typescript'], ['next.js'], ['python'], ['fastapi'], ['rest api']]
O03 [['python'], ['machine learning'], ['deep learning']]
O04 [['devops', 'infrastructure as code', 'automation'], ['networks'], ['gnu/linux', 'linux']]
O05 [['power bi']]
O06 [['html'], ['css'], ['javascript'], ['react', 'angular', 'vue.js'], ['python', 'django', 'node.js']]
O07 []
O08 []
O09 []
O10 []

CVS (v2 skills)
C01 ['power bi', 'python', 'pandas', 'sql', 'postgresql', 'excel', 'basic statistics']
C02 ['web development', 'react', 'node.js', 'git', 'github', 'html', 'css', 'javascript', 'sql', 'mysql']
C03 ['machine learning', 'classification models', 'regression models', 'scikit-learn', 'image classifier', 'pytorch', 'data cleaning', 'pandas', 'python']
C04 ['cybersecurity', 'nmap', 'wireshark', 'linux', 'bash', 'python', 'tcp/ip', 'firewalls']
C05 ['html', 'css', 'java', 'python', 'git']
C06 ['react', 'typescript', 'next.js', 'rest apis'

In [15]:
import json
eval_set = json.load(open("internship-match/data/eval_set.json"))
print(len(eval_set))

24


In [17]:
# Cell 10 is the full system v2: Extract v2 -> Match (best alternative per group) -> Decide (rule). No Gemini call in Match or Decide.

def match_groups(groups, cv_skills):
    out = []
    for g in groups:
        best = max(match(g, cv_skills), key=lambda r: r[2])  # best alternative of the group
        out.append((" / ".join(g), best[1], best[2]))
    return out

def prepare(offer_id, cv_id):
    """Returns (status, res). status is 'error' if an extraction failed, 'empty' if the offer has no requirements."""
    groups, skills = offer_groups[offer_id], cv_skills_v2[cv_id]
    if groups is None or skills is None:
        return "error", []
    groups = [g for g in groups if g]
    if not groups:
        return "empty", []
    if not skills:
        return "ok", [(" / ".join(g), "", 0.0) for g in groups]
    return "ok", match_groups(groups, skills)

def system_v2(offer_id, cv_id):
    status, res = prepare(offer_id, cv_id)
    if status == "error":
        return "ERROR", [], res
    verdict, missing = decide(res)
    return verdict, missing, res

def score_rows(fn):
    rows = []
    for case in eval_set:
        verdict, missing, _ = fn(case["offer"], case["cv"])
        rows.append((case["case"], case["cv"], case["offer"], case["gold_verdict"], verdict, missing,
                     "" if verdict == case["gold_verdict"] else "WRONG"))
    df = pd.DataFrame(rows, columns=["case", "cv", "offer", "gold", "system", "missing", "check"])
    n_ok = (df["check"] == "").sum()
    ref = df[df["gold"] == "refuse"]
    return df, int(n_ok), int((ref["system"] == "refuse").sum()), len(ref)

v2df, v2_ok, v2_ref, n_ref = score_rows(system_v2)
print(v2df.to_string(index=False))
print(f"\nCorrect verdicts: {v2_ok} / {len(v2df)}")
print(f"Correct refusals: {v2_ref} / {n_ref}")
v2df.to_csv("full_system_v2_results.csv", index=False)

 case  cv offer       gold     system                                                  missing check
    1 C01   O01  qualifies  qualifies                                                       []      
    2 C01   O05  qualifies  qualifies                                                       []      
    3 C02   O06  qualifies  qualifies                                                       []      
    4 C06   O02  qualifies  qualifies                                                       []      
    5 C08   O03  qualifies  qualifies                                                       []      
    6 C09   O04  qualifies  qualifies                                                       []      
    7 C02   O05     almost     almost                                               [power bi]      
    8 C03   O03     almost  qualifies                                                       [] WRONG
    9 C04   O04     almost not suited [devops / infrastructure as code / automation, networ

In [24]:
# Cell 11 is the Explain step: Gemini writes the short answer. The refusal sentence is fixed, no Gemini call.

REFUSAL = "Not enough information to judge this offer."

def explain(verdict, missing, res):
    if verdict == "refuse":
        return REFUSAL
    covered = [f"{r} (matched with {s})" for r, s, sc in res if sc >= TAU]
    prompt = f"""You help a student who compared a CV with an internship offer.
Verdict: {verdict}
Requirements covered: {covered}
Requirements missing: {missing}
Write a short answer for the student: 1 or 2 sentences, in English. Say the verdict and name each missing requirement exactly as written above. Do not add any requirement that is not in the lists."""
    return call_model(prompt).strip()

# demo: a covered case, an "almost" case, a vague offer (case numbers in eval_set)
for n in (1, 7, 21):
    case = next(c for c in eval_set if c["case"] == n)
    verdict, missing, res = system_v2(case["offer"], case["cv"])
    print(f"case {n}  {case['cv']} vs {case['offer']}  ->  {verdict}  missing={missing}")
    print("   ", explain(verdict, missing, res), "\n")

case 1  C01 vs O01  ->  qualifies  missing=[]
    Congratulations, you qualify for this internship! You meet all the requirements: Power BI, SQL, and Excel. 

case 7  C02 vs O05  ->  almost  missing=['power bi']
    Your match verdict is almost. The missing requirement is: power bi. 

case 21  C01 vs O07  ->  refuse  missing=[]
    Not enough information to judge this offer. 



In [38]:
import pandas as pd
from IPython.display import display

b = pd.read_csv("internship-match/results/baseline_results.csv")  # Gemini alone
v1 = pd.read_csv("internship-match/results/full_system_results.csv")  # v1 run 1
v2 = pd.read_csv("internship-match/results/full_system_v2_results.csv")

m = b[["case", "cv", "offer", "gold", "gemini"]].rename(columns={"gemini": "system_gemini"})
m = m.merge(v1[["case", "system"]].rename(columns={"system": "system_v1"}), on="case")
m = m.merge(v2[["case", "system"]].rename(columns={"system": "system_v2"}), on="case")
m["gemini_correct"] = m["system_gemini"] == m["gold"]  # True = right
m["v1_correct"] = m["system_v1"] == m["gold"]
m["v2_correct"] = m["system_v2"] == m["gold"]
cols = ["case", "gold", "system_gemini", "system_v1", "system_v2",
        "gemini_correct", "v1_correct", "v2_correct"]
diff = m[(m["system_gemini"] != m["system_v1"]) | (m["system_v1"] != m["system_v2"])]  # not all equal

print("1. Cases where Gemini alone, v1 and v2 differ")
display(diff[cols])

rows = []  # v1 wrong: cases 3, 4
for n in (3, 4):
    c = next(c for c in eval_set if c["case"] == n)
    r = v1[v1["case"] == n].iloc[0]
    rows.append({"case": n, "gold": c["gold_verdict"], "v1": r["system"],
                 "gold_missing": c.get("gold_missing"), "v1_missing": r["missing"]})
print("\n2. Why v1 was wrong (cases 3, 4)")
display(pd.DataFrame(rows))

WRONG_ROWS = {10: ["next.js"], 14: ["javascript", "react / angular / vue.js"]}  # false matches
rows = []  # v2 wrong: cases 10, 14 (Match scores)
for n in (10, 14):
    c = next(c for c in eval_set if c["case"] == n)
    v = system_v2(c["offer"], c["cv"])[0]
    for g, skill, sc in match_groups(offer_groups[c["offer"]], cv_skills_v2[c["cv"]]):
        rows.append({"case": n, "gold": c["gold_verdict"], "v2": v, "requirement": g,
                     "closest_skill": skill, "score": round(sc, 2),
                     "status": "ok" if sc >= TAU else "MISSING",
                     "wrong_row": "<-- WRONG" if g in WRONG_ROWS[n] else ""})
print("\n3. Why v2 was wrong (cases 10, 14): Match scores")
display(pd.DataFrame(rows))

rank = {"not suited": 0, "almost": 1, "qualifies": 2}  # strictness scale
rows = []  # Gemini wrong: how
for _, r in m[~m["gemini_correct"]].iterrows():
    c = next(c for c in eval_set if c["case"] == r["case"])
    kind = "too strict" if rank[r["system_gemini"]] < rank[r["gold"]] else "too lenient"
    rows.append({"case": r["case"], "gold": r["gold"], "gemini": r["system_gemini"],
                 "error": kind, "gold_missing": c.get("gold_missing")})
print("\n4. How Gemini alone was wrong")
display(pd.DataFrame(rows))

1. Cases where Gemini alone, v1 and v2 differ


,case,gold,system_gemini,system_v1,system_v2,gemini_correct,v1_correct,v2_correct
0,1,qualifies,almost,qualifies,qualifies,False,True,True
1,2,qualifies,almost,qualifies,qualifies,False,True,True
2,3,qualifies,qualifies,not suited,qualifies,True,False,True
3,4,qualifies,qualifies,almost,qualifies,True,False,True
4,5,qualifies,almost,qualifies,qualifies,False,True,True
5,6,qualifies,almost,qualifies,qualifies,False,True,True
6,7,almost,not suited,almost,almost,False,True,True
7,8,almost,almost,qualifies,qualifies,True,False,False
8,9,almost,almost,not suited,not suited,True,False,False
9,10,almost,almost,almost,qualifies,True,True,False



2. Why v1 was wrong (cases 3, 4)


,case,gold,v1,gold_missing,v1_missing
0,3,qualifies,not suited,[],"['angular', 'python', 'django']"
1,4,qualifies,almost,[],['rest apis']



3. Why v2 was wrong (cases 10, 14): Match scores


,case,gold,v2,requirement,closest_skill,score,status,wrong_row
0,10,almost,qualifies,react,react,1.00,ok,
1,10,almost,qualifies,typescript,typescript,1.00,ok,
2,10,almost,qualifies,next.js,javascript,0.51,ok,<-- WRONG
3,10,almost,qualifies,python,python,1.00,ok,
4,10,almost,qualifies,fastapi,fastapi,1.00,ok,
5,10,almost,qualifies,rest api,rest apis,0.79,ok,
6,14,not suited,qualifies,html,html,1.00,ok,
7,14,not suited,qualifies,css,css,1.00,ok,
8,14,not suited,qualifies,javascript,html,0.63,ok,<-- WRONG
9,14,not suited,qualifies,react / angular / vue.js,java,0.61,ok,<-- WRONG



4. How Gemini alone was wrong


,case,gold,gemini,error,gold_missing
0,1,qualifies,almost,too strict,[]
1,2,qualifies,almost,too strict,[]
2,5,qualifies,almost,too strict,[]
3,6,qualifies,almost,too strict,[]
4,7,almost,not suited,too strict,[powerbi]
5,16,not suited,almost,too lenient,"[powerbi, sql, excel]"


In [ ]:
# Cell 12 is the ablation: the full system v2 WITHOUT the Decide rule. Extract and Match are the same;
# Gemini gets the requirements, the closest CV skill and the score, and gives the verdict itself (no 0.50 threshold, no counting rule).

def decide_gemini(res):
    lines = "\n".join(f"- {r} -> closest CV skill: {s or '(none)'} (similarity {sc:.2f})" for r, s, sc in res) \
            or "- (no requirements found in the offer)"
    prompt = f"""A student CV was compared with an internship offer. For each requirement of the offer, here is the closest CV skill and a similarity score between 0 and 1:
{lines}

Give one verdict:
- qualifies: the CV covers all requirements of the offer
- almost: exactly one requirement is missing
- not suited: two or more requirements are missing
- refuse: the offer gives too little information to judge

Reply with JSON only, like {{"verdict": "almost", "missing": ["power bi"]}}"""
    data = _json(call_model(prompt)) or {}
    return data.get("verdict", "ERROR"), data.get("missing", [])

def ablation(offer_id, cv_id):
    status, res = prepare(offer_id, cv_id)
    if status == "error":
        return "ERROR", [], res
    verdict, missing = decide_gemini(res)
    return verdict, missing, res

abdf, ab_ok, ab_ref, n_ref = score_rows(ablation)
print(abdf.to_string(index=False))
print(f"\nCorrect verdicts: {ab_ok} / {len(abdf)}")
print(f"Correct refusals: {ab_ref} / {n_ref}")
abdf.to_csv("ablation_results.csv", index=False)

 case  cv offer       gold     system                                                                     missing check
    1 C01   O01  qualifies  qualifies                                                                          []      
    2 C01   O05  qualifies  qualifies                                                                          []      
    3 C02   O06  qualifies  qualifies                                                                          []      
    4 C06   O02  qualifies  qualifies                                                                          []      
    5 C08   O03  qualifies  qualifies                                                                          []      
    6 C09   O04  qualifies  qualifies                                                                          []      
    7 C02   O05     almost     almost                                                                  [power bi]      
    8 C03   O03     almost     almost   

In [ ]:
# Cell 13 prints the final results table, filled with the measured numbers only.

def counts(df):
    ref = df[df["gold"] == "refuse"]
    sysname = "system" if "system" in df.columns else "gemini"
    return int((df["check"] == "").sum()), int((ref[sysname] == "refuse").sum()), len(ref)

rows = [("Gemini alone (baseline)", counts(bdf)),
        ("Full system v1", counts(fdf)),
        ("Full system v2", counts(v2df)),
        ("Without Decide (ablation, on v2)", counts(abdf))]
print("| Configuration | Correct verdicts | Correct refusals |")
print("|---|---|---|")
for name, (ok, r, n) in rows:
    print(f"| {name} | {ok} / {len(eval_set)} | {r} / {n} |")

| Configuration | Correct verdicts | Correct refusals |
|---|---|---|
| Gemini alone (baseline) | 18 / 24 | 4 / 4 |
| Full system v1 | 16 / 24 | 4 / 4 |
| Full system v2 | 18 / 24 | 4 / 4 |
| Without Decide (ablation, on v2) | 21 / 24 | 4 / 4 |


In [ ]:
for n in (8, 9, 10, 12, 13, 14):
    c = next(x for x in eval_set if x["case"] == n)
    _, res = prepare(c["offer"], c["cv"])
    print(n, c["cv"], c["offer"], [(r, s, round(sc, 2)) for r, s, sc in res if sc < 0.8])

8 C03 O03 [('deep learning', 'machine learning', 0.68)]
9 C04 O04 [('devops / infrastructure as code / automation', 'nmap', 0.47), ('networks', 'cybersecurity', 0.43)]
10 C07 O02 [('next.js', 'javascript', 0.51), ('rest api', 'rest apis', 0.79)]
12 C10 O01 [('excel', 'sql', 0.53)]
13 C02 O01 [('power bi', 'git', 0.26), ('excel', 'sql', 0.53)]
14 C05 O06 [('javascript', 'html', 0.63), ('react / angular / vue.js', 'java', 0.61)]
